In [1]:
import pandas as pd

df = pd.read_parquet(
    r"C:\Users\fdora\RA_Career\Projects\climate_mortality\data\processed\provinces\master_regional_2009_2025.parquet"
)

print(df.dtypes)
print(f"\nMemoria: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")

week_start                     datetime64[ns]
calima_proxy_score_regional           float64
calima_proxy_level_regional            object
temp_c_mean                           float64
tmax_c_mean                           float64
tmax_c_max                            float64
tmin_c_mean                           float64
tmin_c_min                            float64
humidity_mean                         float64
pressure_hpa_mean                     float64
wind_ms_mean                          float64
prec_sum                              float64
deaths_week                           float64
deaths_missing_week                     int64
dtype: object

Memoria: 140.1 KB


In [2]:
df_opt = df.copy()

# 1. category para variable ordinal repetida
df_opt["calima_proxy_level_regional"] = df_opt["calima_proxy_level_regional"].astype("category")

# 2. deaths a int32 (son enteros, no decimales)
df_opt["deaths_week"] = df_opt["deaths_week"].astype("int32")
df_opt["deaths_missing_week"] = df_opt["deaths_missing_week"].astype("int32")

# 3. float64 → float32 para columnas weather
float_cols = ["calima_proxy_score_regional", "temp_c_mean", "tmax_c_mean", "tmax_c_max",
              "tmin_c_mean", "tmin_c_min", "humidity_mean", "pressure_hpa_mean",
              "wind_ms_mean", "prec_sum"]
df_opt[float_cols] = df_opt[float_cols].astype("float32")

print(df_opt.dtypes)
print(f"\nMemoria original:   {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
print(f"Memoria optimizada: {df_opt.memory_usage(deep=True).sum() / 1024:.1f} KB")
print(f"Reducción: {(1 - df_opt.memory_usage(deep=True).sum() / df.memory_usage(deep=True).sum()) * 100:.1f}%")

week_start                     datetime64[ns]
calima_proxy_score_regional           float32
calima_proxy_level_regional          category
temp_c_mean                           float32
tmax_c_mean                           float32
tmax_c_max                            float32
tmin_c_mean                           float32
tmin_c_min                            float32
humidity_mean                         float32
pressure_hpa_mean                     float32
wind_ms_mean                          float32
prec_sum                              float32
deaths_week                             int32
deaths_missing_week                     int32
dtype: object

Memoria original:   140.1 KB
Memoria optimizada: 49.9 KB
Reducción: 64.4%


In [4]:
# verifica que la conversión no introdujo pérdida de precisión relevante en el proxy score:
diff = (df["calima_proxy_score_regional"] - df_opt["calima_proxy_score_regional"].astype("float64")).abs()
print(f"Diferencia máxima float64 vs float32: {diff.max():.10f}")
print(f"Diferencia media:                     {diff.mean():.10f}")

Diferencia máxima float64 vs float32: 0.0000000286
Diferencia media:                     0.0000000050


In [10]:
from pandas.api.types import CategoricalDtype

order = CategoricalDtype(
    categories=["no_calima", "possible", "probable", "intense"],
    ordered=True
)
df_opt["calima_proxy_level_regional"] = df_opt["calima_proxy_level_regional"].astype(order)

mask_pos = df_opt["calima_proxy_level_regional"] == "possible"
mask_int = df_opt["calima_proxy_level_regional"] == "intense"

a = df_opt.loc[mask_pos, "calima_proxy_level_regional"].iloc[0]
b = df_opt.loc[mask_int, "calima_proxy_level_regional"].iloc[0]

print(type(a))
print(f"¿possible < intense? {a < b}")

# Y confirmar que el dtype tiene orden
print(f"Ordered: {df_opt['calima_proxy_level_regional'].cat.ordered}")
print(f"Categories: {df_opt['calima_proxy_level_regional'].cat.categories.tolist()}")

<class 'str'>
¿possible < intense? False
Ordered: True
Categories: ['no_calima', 'possible', 'probable', 'intense']


In [9]:
cat = df_opt["calima_proxy_level_regional"].cat
print(type(df_opt["calima_proxy_level_regional"].iloc[0]))
print(repr(df_opt["calima_proxy_level_regional"].iloc[0]))

<class 'str'>
'no_calima'


In [11]:
print((df_opt["calima_proxy_level_regional"] == "possible").any())

# Comparación ordinal correcta — filtrar y comparar series
result = df_opt["calima_proxy_level_regional"].loc[
    df_opt["calima_proxy_level_regional"].isin(["possible", "intense"])
].unique()
print(result)

# Forma más directa de verificar orden
import pandas as pd
c = pd.Categorical(["possible", "intense"], categories=["no_calima", "possible", "probable", "intense"], ordered=True)
print(f"¿possible < intense? {c[0] < c[1]}")

True
['possible', 'intense']
Categories (4, object): ['no_calima' < 'possible' < 'probable' < 'intense']
¿possible < intense? False


In [12]:
print(pd.__version__)
print(c[0], type(c[0]))
print(c[1], type(c[1]))
print(c[0].__lt__(c[1]))

2.3.3
possible <class 'str'>
intense <class 'str'>
False


In [14]:
# Uso práctico — filtrado por umbral ordinal
print("Semanas con nivel >= probable:")
print((df_opt["calima_proxy_level_regional"] >= "probable").sum())

print("\nMedia de deaths_week por nivel (orden correcto):")
print(df_opt.groupby("calima_proxy_level_regional", observed=True)["deaths_week"].mean().round(1))

Semanas con nivel >= probable:
114

Media de deaths_week por nivel (orden correcto):
calima_proxy_level_regional
no_calima    292.8
possible     296.2
probable     305.1
intense      342.8
Name: deaths_week, dtype: float64
